# Lab 04-02 — MMR: trading relevance for diversity

**Track 04 · Retrieval** — plain top-k retrieval returns the k passages closest to the query — and when several of them are near-duplicates (same Wikipedia article, same facts), the context you hand the LLM is redundant. Maximum Marginal Relevance (MMR) re-ranks the candidate pool to trade a little relevance for diversity:

    mmr_score(doc) = lambda * relevance(doc)
                     - (1 - lambda) * max_sim(doc, already_selected)

* `lambda_mult = 1.0` -> pure relevance: identical to plain similarity search
* `lambda_mult = 0.5` -> balanced: relevant but not redundant
* `lambda_mult = 0.0` -> pure diversity: farthest-point sampling from the query

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the plain top-k path, and the MMR sweep all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

This lab runs the same three questions through plain top-5 and through MMR at all three lambdas, then makes the tradeoff visible with two numbers per set: the overlap with the plain top-5 (a relevance proxy) and the mean pairwise cosine distance between the returned passage vectors (a diversity proxy).


## Setup

Two prerequisites must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `sentence-transformers`, `faiss-cpu`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS index (langchain_community)
#   pandas                -> reads the passages/test.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes the deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610, 1604]` are real questions whose answers live inside the subset; `TOP_K = 5` is the plain similarity budget and `MMR_K = 5` the MMR budget (same size, different ranking — so the only variable is diversity); `LAMBDAS = (1.0, 0.5, 0.0)` is the MMR sweep from pure relevance to pure diversity.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
TOP_K = 5  # plain similarity search returns this many hits
MMR_K = 5  # MMR returns this many hits (same budget, different ranking)
LAMBDAS = (1.0, 0.5, 0.0)  # MMR sweep: 1.0 = pure relevance, 0.0 = pure diversity
PREVIEW = 62  # max characters of passage text shown next to each hit
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` pulls the first `n` passages (text + ids) from `passages.parquet`; `load_questions` pulls specific rows by id from `test.parquet`; `preview` flattens a passage onto one line. Two helpers round out the section: `passage_lookup` maps passage id -> text for the demo, and `mean_pairwise_cosine_distance` is the lab's diversity proxy — BGE embeddings are L2-normalized, so cosine similarity is a plain dot product and `1 - cosine` is a proper distance in [0, 2]: 0.0 means the two passages are identical, larger values mean the set is more spread out.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def passage_lookup(texts: list[str], ids: list[int]) -> dict[int, str]:
    """Map passage id -> text, for turning a hit id back into printable text."""
    return dict(zip(ids, texts))


def mean_pairwise_cosine_distance(
    docs: list[Document], id2vec: dict[int, list[float]]
) -> float:
    """Mean 1 - cosine over every pair of returned passage vectors.

    BGE embeddings are L2-normalized, so cosine similarity is a plain dot
    product and 1 - cosine is a proper distance in [0, 2]: 0.0 means the two
    passages are identical, larger values mean the set is more spread out.
    This is the lab's diversity proxy.
    """
    vecs = [id2vec[d.metadata.get("id")] for d in docs]
    total, n = 0.0, 0
    for i in range(len(vecs)):
        for j in range(i + 1, len(vecs)):
            cos = sum(a * b for a, b in zip(vecs[i], vecs[j]))
            total += 1.0 - cos
            n += 1
    return total / n if n else 0.0


## 3. Experiment — embed, index, query plain + MMR at every lambda

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 100 passages once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. The plain path is `similarity_search_with_score_by_vector` at `TOP_K = 5`; the MMR path is the store's NATIVE `max_marginal_relevance_search_by_vector` at every lambda — LangChain's FAISS implements the same greedy MMR formula the shared `src/retrieval/mmr.py` class wraps (there is no MMR retriever class in this stack, so the store method IS the MMR block). `run_experiment` returns a dict of artifacts instead of printing, so the demo and the gate read the same run.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed, index, query plain + MMR at every lambda; returns
#    every artifact the demo and the verification gate need (no re-computation
#    between the two paths)
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed the whole subset once (batched) + each question once ---------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    question_texts = [qtext for _, qtext in questions]
    query_vecs = [embedder.embed_query(q) for q in question_texts]

    # --- Build the FAISS index (in-memory) ----------------------------------
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    # from_embeddings builds the index from the precomputed vectors (so the
    # embed step and the index step stay separately timed) while keeping the
    # real embedder attached — store.embed_query() works for the retriever.
    t0 = time.perf_counter()
    store = FAISS.from_embeddings(
        list(zip(passage_texts, passage_vecs)),
        embedding=embedder,
        metadatas=[{"id": pid} for pid in passage_ids],
    )
    index_s = time.perf_counter() - t0

    id2vec = dict(zip(passage_ids, passage_vecs))

    # --- Per question: plain top-K + MMR at every lambda --------------------
    results = []
    for qi, (qid, qtext) in enumerate(questions):
        plain = store.similarity_search_with_score_by_vector(query_vecs[qi], k=TOP_K)
        plain_ids = [d.metadata.get("id") for d, _ in plain]
        plain_set = set(plain_ids)

        # The MMR block: the store's native max_marginal_relevance_search
        # (greedy lambda*relevance - (1-lambda)*max_sim-to-selected), the same
        # formula the repo's MMRRetriever wraps.
        mmr: dict[float, list[Document]] = {}
        overlap: dict[float, int] = {}
        dist: dict[float, float] = {}
        for lam in LAMBDAS:
            docs = store.max_marginal_relevance_search_by_vector(
                query_vecs[qi], k=MMR_K, lambda_mult=lam
            )
            mmr[lam] = docs
            overlap[lam] = len(plain_set & {d.metadata.get("id") for d in docs})
            dist[lam] = mean_pairwise_cosine_distance(docs, id2vec)

        results.append(
            {
                "qid": qid,
                "qtext": qtext,
                "plain": plain,
                "plain_ids": plain_ids,
                "plain_dist": mean_pairwise_cosine_distance(
                    [d for d, _ in plain], id2vec
                ),
                "mmr": mmr,
                "overlap": overlap,
                "dist": dist,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "index_s": index_s,
        "results": results,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with the three questions and embed/index timings; the tradeoff at a glance — per question, the plain top-5 ids with their spread, and for each lambda the MMR ids, the overlap with plain (relevance proxy) and the spread (diversity proxy); passage previews for the first question through plain top-5 and MMR at every lambda; then a takeaway framing MMR as a single knob on the same retriever.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    passage_lk = passage_lookup(exp["passage_texts"], exp["passage_ids"])

    print("=" * 66)
    print("Lab 02 — MMR: trading relevance for diversity")
    print(f"{BGE_MODEL_NAME} | FAISS flat-L2 | in-memory only")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions from test.parquet:")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] Embed + index:")
    print(f"    embedded {exp['indexed']} passages in {exp['embed_s']:.2f}s (dim {exp['dim']})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s")

    print(f"\n[3] The tradeoff at a glance — plain top-{TOP_K} vs MMR at each lambda:")
    print(f"    overlap = |plain set ∩ mmr set| (relevance proxy, higher = closer to plain)")
    print(f"    spread  = mean pairwise 1-cosine distance of the returned set (diversity proxy)")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["qtext"]}"')
        print(f"      plain top-{TOP_K}: ids={r['plain_ids']}  spread={r['plain_dist']:.4f}")
        for lam in LAMBDAS:
            ids = [d.metadata.get("id") for d in r["mmr"][lam]]
            print(f"      mmr @{lam:<3} : ids={ids}  overlap={r['overlap'][lam]}/{TOP_K}  spread={r['dist'][lam]:.4f}")

    print(f"\n[4] Passage previews — Q[{exp['results'][0]['qid']}] "
          f"\"{exp['results'][0]['qtext']}\" (score = squared L2, LOWER = more similar):")
    r0 = exp["results"][0]
    print(f'\n    plain top-{TOP_K}:')
    for doc, score in r0["plain"]:
        pid = doc.metadata.get("id", "?")
        print(f"      {score:8.4f}  [passage {pid}] {preview(doc.page_content)}")
    for lam in LAMBDAS:
        print(f"\n    MMR lambda_mult={lam} (k={MMR_K}):")
        for rank, doc in enumerate(r0["mmr"][lam], 1):
            pid = doc.metadata.get("id", "?")
            print(f"      {rank}. [passage {pid}] {preview(doc.page_content)}")
    print("      ^ lambda 1.0 keeps the plain ranking; as lambda drops, MMR")
    print("        swaps near-duplicate passages for spread-out ones.")

    print("\n[5] Takeaway")
    print("    MMR is a single knob on the same retriever: lambda_mult=1.0 is")
    print("    plain similarity, 0.0 is farthest-point diversity, and 0.5 sits")
    print("    in between. The overlap column shows the relevance you give up;")
    print("    the spread column shows the diversity you buy. For a RAG context")
    print("    window, a diverse top-5 often beats five near-duplicates of the")
    print("    same fact — but pure diversity (0.0) can drift off-topic, which")
    print("    is why 0.5 is the usual default.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: embedding dimension 768 and exactly `N_PASSAGES` passages indexed; plain retrieval returns exactly `TOP_K` scored hits per question; MMR at every lambda returns exactly `MMR_K` distinct documents (no dupes); lambda 1.0 reproduces the plain top-1 (pure relevance — verified empirically as an exact match); overlap at lambda 0.0 is strictly below overlap at 1.0 for every question, and lambda 0.5 sits strictly between; the diversity proxy moves the other way (`spread(0.0) > spread(1.0)`); and each question's plain top-1 is on-topic. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Dimension and count match the model / subset.
    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Plain retrieval returns exactly TOP_K scored hits per question.
    checks.append(
        ("each question returns exactly TOP_K plain hits",
         all(len(r["plain"]) == TOP_K for r in exp["results"]))
    )

    # MMR at every lambda returns exactly MMR_K distinct documents (no dupes).
    mmr_distinct = all(
        len(docs) == MMR_K and len({d.metadata.get("id") for d in docs}) == MMR_K
        for r in exp["results"]
        for docs in r["mmr"].values()
    )
    checks.append((f"MMR at every lambda returns {MMR_K} distinct docs", mmr_distinct))

    # lambda_mult=1.0 is pure relevance: it must reproduce the plain
    # similarity ranking (BGE vectors are L2-normalized, so cosine and
    # squared-L2 order identically). Verified empirically: exact match.
    top1_agree = all(
        r["plain_ids"][0] == r["mmr"][1.0][0].metadata.get("id")
        for r in exp["results"]
    )
    checks.append(("MMR lambda=1.0 top-1 == plain top-1 (pure relevance)", top1_agree))

    # Diversity grows as lambda drops: the MMR set drifts away from the plain
    # set. Verified empirically: overlap 1 < 2 < 5 for every question.
    overlap_mono = all(
        r["overlap"][0.0] < r["overlap"][1.0] for r in exp["results"]
    )
    checks.append(("overlap(plain, MMR@0.0) < overlap(plain, MMR@1.0) for every question", overlap_mono))

    overlap_mid = all(
        r["overlap"][0.0] < r["overlap"][0.5] < r["overlap"][1.0]
        for r in exp["results"]
    )
    checks.append(("overlap at lambda=0.5 sits strictly between 0.0 and 1.0", overlap_mid))

    # The diversity proxy moves the other way: lower lambda -> more spread.
    # Verified empirically: spread(0.0) > spread(1.0) for every question.
    dist_mono = all(
        r["dist"][0.0] > r["dist"][1.0] for r in exp["results"]
    )
    checks.append(("mean pairwise distance MMR@0.0 > MMR@1.0 (more diverse)", dist_mono))

    # Content checks: each question's plain top-1 is on-topic.
    q1606_top = exp["results"][0]["plain"][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))
    q1610_top = exp["results"][1]["plain"][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))
    q1604_top = exp["results"][2]["plain"][0][0].page_content.lower()
    checks.append(("Q1604 top-1 mentions Uruguay", "uruguay" in q1604_top))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A couple of minutes of embedding + index build — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The overlap/spread table for all three questions across plain top-5 and the three MMR lambdas, plus passage previews for the first question.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
